# UIT DSC 2026 LegalIR - Step 3 Tune Chunk-BM25

Tune Chunk-BM25 on dev before moving to dense retrieval. This step is CPU-bound; GPU is not required.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

DATA_ROOT = Path('/kaggle/input/datasets/bowboochua9/stnhdscduaiti26')
PUBLIC_FILE = Path('/kaggle/input/datasets/ttdatto/uit-dsc26/LegalIR - Public Test/public-official.json')
OUTPUT_DIR = Path('/kaggle/working/step3')

SCRIPT_CANDIDATES = [
    Path('/kaggle/working/legalir_step3_tune_bm25.py'),
    Path('/kaggle/working/step3/legalir_step3_tune_bm25.py'),
    Path('/kaggle/input/dscuit2026-code/task1/pipeline/step3/legalir_step3_tune_bm25.py'),
    Path('/kaggle/input/dscuit2026/task1/pipeline/step3/legalir_step3_tune_bm25.py'),
    Path('legalir_step3_tune_bm25.py'),
    Path('task1/pipeline/step3/legalir_step3_tune_bm25.py'),
]
SCRIPT_PATH = next((p for p in SCRIPT_CANDIDATES if p.exists()), None)
if SCRIPT_PATH is None:
    raise FileNotFoundError('Cannot find legalir_step3_tune_bm25.py. Add it as a Kaggle utility script or attach this repo as a dataset.')

print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('SCRIPT_PATH:', SCRIPT_PATH)
assert (DATA_ROOT / 'chunks.jsonl').exists()
assert (DATA_ROOT / 'train_split.json').exists()
assert (DATA_ROOT / 'dev_split.json').exists()

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
report = json.loads((OUTPUT_DIR / 'reports' / 'run_report.json').read_text(encoding='utf-8'))
summary = json.loads((OUTPUT_DIR / 'metrics' / 'ablation_summary.json').read_text(encoding='utf-8'))
print(json.dumps({
    'best_trial': report['best_trial'],
    'best_config': report['best_config'],
    'best_dev_macro': report['best_dev_macro'],
    'top_trials': [
        {'name': row['name'], 'macro': row['metrics']['macro']} for row in summary[:5]
    ],
}, ensure_ascii=False, indent=2))

## Optional public submission

Run this only when Step 3 improves dev enough to spend one public submission attempt.

In [ ]:
MAKE_PUBLIC_SUBMISSION = False

if MAKE_PUBLIC_SUBMISSION:
    assert PUBLIC_FILE.exists()
    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        '--data-root', str(DATA_ROOT),
        '--best-config-file', str(OUTPUT_DIR / 'configs' / 'best_config.json'),
        '--public-file', str(PUBLIC_FILE),
        '--output-dir', str(OUTPUT_DIR / 'public_run'),
        '--predict-public',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
    print('Submission ZIP:', OUTPUT_DIR / 'public_run' / 'submission' / 'submission.zip')
else:
    print('Skipped public submission. Set MAKE_PUBLIC_SUBMISSION = True when ready.')

In [ ]:
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), path.stat().st_size)